In [0]:
import requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql import DataFrame
from functools import reduce
from delta.tables import DeltaTable
import requests, zipfile, io
import re
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
from datetime import datetime
import pprint
import os
import json
import time

## CVM - Fundos de Investimento, Classes e Subclasses de Cotas

### 1. Verificando os arquivos

Segundo o Site da CVM a atualização dos arquivos ocorre de terça a sábado, às 08:00h, com a posição cadastral dos fundos até as 23:59h do dia anterior. Sendo assim conseguimos economizar no processamento nesses dias 

In [0]:

lista_registros = dict()
BASE_URL_DATA_CRUSE = "https://dados.cvm.gov.br/dados/FI/CAD/DADOS/registro_fundo_classe.zip"


# 0 = segunda, 6 = domingo
if hoje.weekday() in [0, 6]:
    print("Sem ingestão nas segundas e domingos")
else:
    # BAIXA OS DADOS (RAW)
    response = requests.get(BASE_URL_DATA_CRUSE)
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        for file_name in z.namelist():
            if file_name.endswith(".csv"):
                output_path = os.path.join("/Volumes/workspace/case_spark_cvm/raw/cvm_registros/", file_name)

                with z.open(file_name) as source, open(output_path, "wb") as target:
                    target.write(source.read())

                print(f"  → Extraído: {output_path}")
                lista_registros[output_path.split('/')[-1].split(".csv")[0]] = output_path

### 2. Salvar em camada Bronze Particionada

In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

if hoje.weekday() in [6, 0]: 
    print("Sem processamento nas segundas e Domingos")

else:
    for name_path, caminho in lista_registros.items():

        output_path = f"/Volumes/workspace/case_spark_cvm/bronze/{name_path}_cvm/"
        print(f'Processamento: {name_path}')
        print(f"Output: {output_path}")

        df = spark.read.csv(caminho, sep=';', header=True)

        df = df.withColumn(
                "data_processamento",
            f.date_format(f.current_date(), "yyyyMMdd").cast("int")
        )

        df.write \
        .mode('overwrite') \
        .option("replaceWhere", f"data_processamento = {data_proc}")\
        .partitionBy('data_processamento')\
        .format('delta')\
        .save(output_path)

In [0]:
display(df)